# Notebook Kaggle : Pipeline NLP CREMMA Medieval

Ce notebook clone le code de la branche `nlp-pipeline`, recupere les donnees HTR depuis S3,
puis execute la pipeline NLP reelle via `nlp_pipeline/nlp_cli.py` :
validate -> eda -> review-queue -> normalize-contract -> correct -> ablation -> split.

## 1. Cloner le repo et installer les dependances

In [ ]:
!git clone https://github.com/Loulou441/htr-cremma-medieval-2026.git
%cd htr-cremma-medieval-2026
!git checkout nlp-pipeline
!pip install -q jsonschema transformers sentencepiece boto3

## 2. Recuperer les donnees HTR depuis S3

Les sorties HTR (`data/nlp_output/`) et le lexique sont stockes sur `s3://htr-cremma-medieval/nlp/`.
Renseignez vos credentials AWS dans les **Kaggle Secrets** (Add-ons > Secrets) sous les noms
`AWS_ACCESS_KEY_ID` et `AWS_SECRET_ACCESS_KEY`.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = "eu-west-3"

!aws s3 sync s3://htr-cremma-medieval/nlp/output/ data/nlp_output/
!aws s3 sync s3://htr-cremma-medieval/nlp/lexique/ data/lexique/
!aws s3 cp s3://htr-cremma-medieval/nlp/dictionary/dictionnaire_ancien_francais.json data/dictionary/dictionnaire_ancien_francais.json

## 3. Verifier que le CLI est accessible

In [ ]:
from pathlib import Path

project_root = Path(".").resolve()
cli_path = project_root / "nlp_pipeline" / "nlp_cli.py"
print("CLI exists:", cli_path.exists())

sample_docs = sorted((project_root / "data" / "nlp_output").rglob("*.json"))
print("Sample HTR contracts found:", len(sample_docs))
sample_doc = sample_docs[0] if sample_docs else None
print("Using sample:", sample_doc)

## 4. Lancer la pipeline NLP etape par etape

Chaque commande est appelee en sous-processus pour eviter tout souci de `sys.path`.

In [ ]:
import subprocess


def run_cli(*args):
    cmd = ["python", str(cli_path), *args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print("$ ", " ".join(cmd))
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

In [ ]:
# 4.1 Validation du contrat HTR
run_cli("validate", "--input", str(sample_doc))

In [ ]:
# 4.2 EDA : statistiques de confiance et longueur de ligne
run_cli("eda", "--input", str(sample_doc), "--output", "data/eda_report.json")

In [ ]:
# 4.3 File de relecture : tri par confiance
run_cli(
    "review-queue",
    "--input", str(sample_doc),
    "--csv-output", "data/review/review_queue.csv",
    "--json-output", "data/review/review_buckets.json",
)

In [ ]:
# 4.4 Normalisation du francais medieval (u/v, i/j, abreviations)
run_cli(
    "normalize-contract",
    "--input", str(sample_doc),
    "--output", "data/contracts/contract_normalized.json",
)

In [ ]:
# 4.5 Correction guidee par confiance (heuristique par defaut, sans MLM)
run_cli(
    "correct",
    "--input", str(sample_doc),
    "--output", "data/contracts/contract_corrected.json",
    "--log-output", "data/review/correction_log.jsonl",
)

In [ ]:
# 4.5bis Variante avec modele de langue masque CamemBERT (plus lent, GPU recommande)
# run_cli(
#     "correct",
#     "--input", str(sample_doc),
#     "--output", "data/contracts/contract_corrected_mlm.json",
#     "--log-output", "data/review/correction_log_mlm.jsonl",
#     "--mlm-model", "almanach/camembert-base",
#     "--mlm-device", "auto",
# )

## 5. Evaluer le gain de normalisation (CER avant/apres)

Necessite un CSV avec une colonne de reference et une colonne d'hypothese HTR brute.

In [ ]:
import csv
import json

contract = json.loads(Path(sample_doc).read_text(encoding="utf-8"))
rows = [
    {"reference": line["text"], "text": line["text"]}
    for page in contract.get("pages", [])
    for line in page.get("lines", [])
]
ablation_csv = Path("data/review/ablation_sample.csv")
with ablation_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["reference", "text"])
    writer.writeheader()
    writer.writerows(rows)

run_cli("ablation", "--csv-input", str(ablation_csv))

## 6. Push des resultats vers S3 (optionnel)

In [ ]:
sync_script = project_root / "nlp_pipeline" / "sync_to_s3.py"
subprocess.run(["python", str(sync_script)], capture_output=True, text=True)